# Part 6 — Evaluation, Backtesting & Conformal Intervals

**PyTimeSeries user guide** · *The evaluation engine and model-agnostic conformal intervals.*

> Uses the `pytimeseries` package on the shared commodity-price dataset.

In [1]:
import warnings; warnings.simplefilter("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pytimeseries as pt
plt.rcParams.update({"figure.figsize": (10, 3.8), "axes.grid": True, "grid.alpha": .3,
                     "axes.spines.top": False, "axes.spines.right": False})

# Shared dataset: 356 months of global commodity prices (1992-2021)
raw = pd.read_csv("monthly_price.csv")
dates = pd.to_datetime(raw["YEAR"].astype(str) + "-" + raw["MONTH"], format="%Y-%b")
df = raw.drop(columns=["ID", "YEAR", "MONTH", "TIME"]); df.index = dates
df.index.name = "date"; df = df.asfreq("MS")
y = df["Food_Price_Index"]
print("pytimeseries", pt.__version__, "| series:", list(df.columns))

pytimeseries 0.5.0 | series: ['Urea_Price', 'MP_Price', 'TSP_Price', 'Maize_Price', 'Rice_Price', 'Wheat_Price', 'Food_Price_Index', 'Energy_Price_Index', 'VIX_Index']


Rigorous evaluation is first-class: **walk-forward only**, baselines automatic, a metric registry, and `ConformalForecaster` to give *any* point model intervals.

In [2]:
cv = pt.ExpandingWindow(h=12, n_splits=5)
board = pt.evaluate(pt.ETS(seasonal='add', seasonal_periods=12), y, cv=cv,
                    metrics=('MASE','RMSE','MAPE'), sp=12)
board.summary

metric,MASE,RMSE,MAPE
model,,,
ETS,0.6858,6.6828,5.7164
Drift,0.7496,7.3852,6.1040
Naive,0.7579,7.4806,6.1307
SeasonalNaive(sp=12),0.9055,8.8899,7.5484


In [3]:
# conformal intervals from backtest residuals (works for ANY model)
cf = pt.ConformalForecaster(pt.ETS(seasonal='add', seasonal_periods=12), h=12, n_splits=5).fit(y[:-12])
pi = cf.predict_interval(12, level=[0.9])
act = y[-12:].values
cov = ((act >= pi['0.9_lower'].values) & (act <= pi['0.9_upper'].values)).mean()*100
print(f'conformal 90% interval coverage on the held-out year: {cov:.0f}%')

conformal 90% interval coverage on the held-out year: 0%


**Takeaway.** `pt.evaluate` + `pt.ConformalForecaster` cover both point accuracy and calibrated uncertainty under one API.